# Thai Air Intelligence — PM2.5 v5.6.4 DB-only Trainer

Canonical Colab for the same **v5.6.4** Python entrypoint used by the scheduled training workflow. It trains the 20 province-local LightGBM residual regressors and the pooled Random Forest classifier directly from **Supabase `training_daily_summary_v3`**.

The tuning batch is guarded: LightGBM alternatives may improve D+1 only when Validation also preserves D+2–D+7, while Random Forest alternatives focus on Classes 2–3 only when Classes 4–5 and the overall validation metrics remain inside their guardrails. Test is evaluation-only and is never used to select a tuning profile.

This notebook makes **zero Open-Meteo/CAMS archive network reads**. Historical archive data are already persisted in Supabase with lineage. `hotspot_count` and `total_frp` remain gated for the next feature schema and are not silently introduced into v5.6.4.

This checked-in notebook is **safe-shadow** by default. It shows exact per-province regression gate failures, D+1…D+7 MAE, per-class Precision/Recall/F1, and the five-class confusion matrix.


In [ ]:
# 1. Clone the reviewed repository and install the pinned training stack
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/kzabCde/THAI-AIR-INTELLIGENCE-LITE.git"
REPOSITORY_DIRECTORY = Path("/content/THAI-AIR-INTELLIGENCE-LITE")
if not REPOSITORY_DIRECTORY.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIRECTORY)], check=True)
os.chdir(REPOSITORY_DIRECTORY)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", "training/requirements.txt"], check=True)
print("Repository and training dependencies are ready")


In [ ]:
# 2. Reviewed v5.6.4 execution contract
from pathlib import Path

APPROVED_CODE_SHA = "d096bb9781b7794f49157403b41e985522a9e6e6"
TRAINER_VERSION = "5.6.4"
SOURCE_OF_TRUTH = "training_daily_summary_v3"
FEATURE_VERSION = "daily-pooled-v1"
REQUIRED_PROVINCES = 20
REGRESSION_SKILL_GATE = 0.045
DRY_RUN = True
ALLOW_PRODUCTION_PROMOTION = False
ARTIFACT_DIRECTORY = Path("/content/pm25_v5_6_4_artifacts")
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

if len(APPROVED_CODE_SHA) != 40:
    raise ValueError("APPROVED_CODE_SHA must be a full reviewed commit SHA")
if not DRY_RUN and not ALLOW_PRODUCTION_PROMOTION:
    raise ValueError("Set ALLOW_PRODUCTION_PROMOTION=True before a non-dry run")

print({"trainer_version": TRAINER_VERSION, "approved_code_sha": APPROVED_CODE_SHA, "source_of_truth": SOURCE_OF_TRUTH, "network_archive_reads": 0, "feature_version": FEATURE_VERSION, "required_provinces": REQUIRED_PROVINCES, "regression_skill_gate": REGRESSION_SKILL_GATE, "dry_run": DRY_RUN})


In [ ]:
# 3. Load server-side Supabase secrets and verify the canonical trainer
import os
import sys
from google.colab import userdata

def _secret(name):
    try:
        value = (userdata.get(name) or "").strip()
    except Exception:
        value = ""
    if not value:
        raise ValueError(f"Missing Colab Secret: {name}")
    return value

os.environ["SUPABASE_URL"] = _secret("SUPABASE_URL")
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = _secret("SUPABASE_SERVICE_ROLE_KEY")
if str(REPOSITORY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIRECTORY))
from training.train_models_pm25_v5_6_4 import _preflight
preflight = _preflight()
if preflight["source_of_truth"] != SOURCE_OF_TRUTH or preflight["active_feature_version"] != FEATURE_VERSION or preflight["provinces"] != REQUIRED_PROVINCES or preflight["network_archive_reads"] != 0:
    raise RuntimeError(f"Unexpected v5.6.4 preflight contract: {preflight}")
print(preflight)


In [ ]:
# 4. Run the canonical v5.6.4 trainer once
command = [sys.executable, "-u", "-m", "training.train_models_pm25_v5_6_4", "--artifact-dir", str(ARTIFACT_DIRECTORY)]
if DRY_RUN:
    command.append("--dry-run")
completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(f"v5.6.4 training failed with exit code {completed.returncode}")
print("v5.6.4 training completed")


In [ ]:
# 5. Exact regression eligibility by province — identify every strict failure
import json
import pandas as pd
from IPython.display import display
regression_csv = ARTIFACT_DIRECTORY / "v5_6_4_regression_eligibility.csv"
regression_json = ARTIFACT_DIRECTORY / "v5_6_4_regression_eligibility.json"
if not regression_csv.exists() or not regression_json.exists():
    raise RuntimeError("Regression eligibility report is missing")
regression_eligibility = pd.read_csv(regression_csv).sort_values("skill_vs_persistence")
regression_audit = json.loads(regression_json.read_text(encoding="utf-8"))
display(regression_eligibility[["province_id", "eligible", "skill_percent", "required_skill_percent", "margin_to_gate_percentage_points", "mae", "baseline_mae", "eligibility_reasons", "tuning_attempted", "tuning_selected", "selected_profile"]])
print("Strict regression failures:", regression_audit["failed_provinces"])


In [ ]:
# 6. Chart — regression Skill vs Persistence by province
import matplotlib.pyplot as plt
ordered = regression_eligibility.sort_values("skill_percent")
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(ordered["province_id"], ordered["skill_percent"])
ax.axhline(REGRESSION_SKILL_GATE * 100.0, linestyle="--", label="Strict gate 4.5%")
ax.set_title("v5.6.4 Regression Skill vs Persistence by Province")
ax.set_xlabel("Province")
ax.set_ylabel("MAE skill vs persistence (%)")
ax.tick_params(axis="x", rotation=60)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 7. Classification focus report — Classes 2–3 with Class 4–5 protection
classification_path = ARTIFACT_DIRECTORY / "v5_6_4_classification_focus.json"
chart_data_path = ARTIFACT_DIRECTORY / "v5_6_4_chart_data.json"
promotion_path = ARTIFACT_DIRECTORY / "monthly_promotion.json"
if not classification_path.exists() or not chart_data_path.exists() or not promotion_path.exists():
    raise RuntimeError("Classification/chart/promotion audit files are missing")
classification_audit = json.loads(classification_path.read_text(encoding="utf-8"))
chart_data = json.loads(chart_data_path.read_text(encoding="utf-8"))
promotion = json.loads(promotion_path.read_text(encoding="utf-8"))
class_metrics = pd.DataFrame(chart_data["classification_by_class"])
horizon_metrics = pd.DataFrame(chart_data["regression_mae_by_horizon"])
display(class_metrics)
print({"selected_rf_profile": classification_audit.get("profile"), "class_2_3_mean_f1": classification_audit.get("class_2_3_mean_f1"), "critical_class_recall": classification_audit.get("critical_class_recall"), "candidate_run_id": promotion.get("run_id"), "champion_run_id": promotion.get("champion_run_id"), "promoted": promotion.get("promoted"), "activation_status": promotion.get("activation_status")})


In [ ]:
# 8. Chart — regression MAE from D+1 through D+7
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(horizon_metrics["horizon"], horizon_metrics["mae"], marker="o")
ax.set_title("v5.6.4 Regression MAE by Forecast Horizon")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("MAE (µg/m³)")
ax.set_xticks(horizon_metrics["horizon"])
fig.tight_layout()
plt.show()


In [ ]:
# 9. Chart — Precision, Recall and F1 for all five classes
metric_names = ["precision", "recall", "f1"]
x = range(len(class_metrics))
width = 0.24
fig, ax = plt.subplots(figsize=(9, 5))
for offset, metric_name in zip((-width, 0.0, width), metric_names):
    ax.bar([value + offset for value in x], class_metrics[metric_name], width=width, label=metric_name.title())
ax.set_title("v5.6.4 Random Forest Metrics by Air-Quality Class")
ax.set_xlabel("Class")
ax.set_ylabel("Score")
ax.set_xticks(list(x), [str(value) for value in class_metrics["class_id"]])
ax.set_ylim(0.0, 1.0)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 10. Chart — five-class confusion matrix
import numpy as np
confusion = np.asarray(chart_data["classification_confusion_matrix"], dtype=int)
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(confusion)
fig.colorbar(image, ax=ax)
ax.set_title("v5.6.4 Random Forest Confusion Matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_xticks(range(5), ["1", "2", "3", "4", "5"])
ax.set_yticks(range(5), ["1", "2", "3", "4", "5"])
for row in range(confusion.shape[0]):
    for column in range(confusion.shape[1]):
        ax.text(column, row, str(confusion[row, column]), ha="center", va="center")
fig.tight_layout()
plt.show()


In [ ]:
# 11. Final champion/challenger decision and tuning audit
summary = {"trainer": TRAINER_VERSION, "source_of_truth": SOURCE_OF_TRUTH, "network_archive_reads": 0, "strict_regression_failures": regression_audit["failed_provinces"], "classification_profile": classification_audit.get("profile"), "candidate_run_id": promotion.get("run_id"), "champion_run_id": promotion.get("champion_run_id"), "candidate_ready": promotion.get("candidate_ready"), "promoted": promotion.get("promoted"), "activation_status": promotion.get("activation_status"), "promotion": promotion.get("promotion")}
display(pd.json_normalize(summary, sep="."))


In [ ]:
# 12. Download the compact audit bundle
import shutil
from google.colab import files
archive = shutil.make_archive("/content/pm25_v5_6_4_audit_bundle", "zip", root_dir=ARTIFACT_DIRECTORY)
print("Audit bundle ready:", archive)
files.download(archive)
